In [1]:
from pydantic import BaseModel
import random

In [2]:
class RSAPublicKey(BaseModel):
    n: int
    e: int

class RSAPrivateKey(BaseModel):
    n: int
    d: int

class RSA:
    our_public_key: RSAPublicKey
    our_private_key: RSAPrivateKey

    def __init__(self, our_public_key: RSAPublicKey, our_private_key: RSAPrivateKey):
        self.our_public_key = our_public_key
        self.our_private_key = our_private_key

    def _encrypt_big_message(self, message: bytes, recipient_public_key: RSAPublicKey) -> bytes:
        modulus_size = (recipient_public_key.n.bit_length() + 7) // 8
        # Reserve 1 byte for the payload length, ensuring the chunk integer remains < modulus n
        chunk_size = modulus_size - 2 
        
        encrypted_chunks = []
        for i in range(0, len(message), chunk_size):
            piece = message[i:i + chunk_size]
            # Pack the chunk: [payload] + [1 byte length indicator]
            chunk = piece + bytes([len(piece)])
            
            # Encrypt the chunk and pad it to exactly modulus_size bytes
            encrypted_chunk = self._encrypt_single_block(chunk, recipient_public_key)
            encrypted_chunks.append(encrypted_chunk)
            
        return b''.join(encrypted_chunks)
    
    def _encrypt_single_block(self, message: bytes, recipient_public_key: RSAPublicKey) -> bytes:
        modulus_size = (recipient_public_key.n.bit_length() + 7) // 8
        message_int = int.from_bytes(message, byteorder='big')
        encrypted_int = pow(message_int, recipient_public_key.e, recipient_public_key.n)
        # Using a fixed size automatically pads with leading zeros
        return encrypted_int.to_bytes(modulus_size, byteorder='big')

    def encrypt(self, message: bytes, recipient_public_key: RSAPublicKey) -> bytes:
        modulus_size = (recipient_public_key.n.bit_length() + 7) // 8
        # Use chunking if the message is too large to fit safely in a single block
        if len(message) >= modulus_size - 1:
            return self._encrypt_big_message(message, recipient_public_key)
        return self._encrypt_single_block(message, recipient_public_key)

    def _decrypt_big_message(self, encrypted_message: bytes) -> bytes:
        modulus_size = (self.our_private_key.n.bit_length() + 7) // 8
        chunk_size = modulus_size
        
        decrypted_chunks = []
        for i in range(0, len(encrypted_message), chunk_size):
            chunk = encrypted_message[i:i + chunk_size]
            
            # Decrypt the block to the maximum plain block size (modulus_size - 1)
            decrypted_block = self._decrypt_single_block(chunk, modulus_size - 1)
            
            # The last byte indicates the original length of the payload in this chunk
            length = decrypted_block[-1]
            payload = decrypted_block[len(decrypted_block) - 1 - length : len(decrypted_block) - 1]
            decrypted_chunks.append(payload)
            
        return b''.join(decrypted_chunks)
    
    def _decrypt_single_block(self, encrypted_message: bytes, out_size: int) -> bytes:
        encrypted_int = int.from_bytes(encrypted_message, byteorder='big')
        decrypted_int = pow(encrypted_int, self.our_private_key.d, self.our_private_key.n)
        return decrypted_int.to_bytes(out_size, byteorder='big')

    def decrypt(self, encrypted_message: bytes) -> bytes:
        modulus_size = (self.our_private_key.n.bit_length() + 7) // 8
        if len(encrypted_message) > modulus_size:
            return self._decrypt_big_message(encrypted_message)
            
        # For small single messages (e.g. "Hello"), decrypt directly without metadata unpacking
        encrypted_int = int.from_bytes(encrypted_message, byteorder='big')
        decrypted_int = pow(encrypted_int, self.our_private_key.d, self.our_private_key.n)
        return decrypted_int.to_bytes((decrypted_int.bit_length() + 7) // 8, byteorder='big')

    def generate_keypair(self, bit_length: int) -> tuple[RSAPublicKey, RSAPrivateKey]:
        def is_probably_prime(n: int, k: int = 5) -> bool:
            """Use Miller-Rabin primality test to check if n is probably prime."""
            if n <= 1:
                return False
            if n <= 3:
                return True
            if n % 2 == 0:
                return False
            
            # Write n-1 as d*2^r
            r, d = 0, n - 1
            while d % 2 == 0:
                d //= 2
                r += 1
            
            # Witness loop
            for _ in range(k):
                a = random.randint(2, n - 2)
                x = pow(a, d, n)
                if x == 1 or x == n - 1:
                    continue
                for _ in range(r - 1):
                    x = pow(x, 2, n)
                    if x == n - 1:
                        break
                else:
                    return False
            return True
        
        def generate_large_prime(bit_length: int) -> int:
            """Generate a large prime number of specified bit length."""
            while True:
                # Generate a random odd integer of the specified bit length
                p = random.getrandbits(bit_length) | 1 | (1 << (bit_length - 1))
                if is_probably_prime(p):
                    return p
        
        p = generate_large_prime(bit_length // 2)
        q = generate_large_prime(bit_length // 2)

        n = p * q
        phi = (p - 1) * (q - 1)
        e = 65537  # Common choice for e
        d = pow(e, -1, phi)  # Modular inverse of e mod phi

        public_key = RSAPublicKey(n=n, e=e)
        private_key = RSAPrivateKey(n=n, d=d)
        return public_key, private_key


def generate_rsa_keypair(bit_length: int) -> tuple[RSAPublicKey, RSAPrivateKey]:
    from Crypto.PublicKey import RSA
    key = RSA.generate(bit_length)
    public_key = RSAPublicKey(n=key.n, e=key.e)
    private_key = RSAPrivateKey(n=key.n, d=key.d)
    return public_key, private_key


In [3]:
keypair_sender = generate_rsa_keypair(2048)
rsa_sender = RSA(our_public_key=keypair_sender[0], our_private_key=keypair_sender[1])

keypair_alice = generate_rsa_keypair(2048)
rsa_alice = RSA(our_public_key=keypair_alice[0], our_private_key=keypair_alice[1])

keypair_bob = generate_rsa_keypair(2048)
rsa_bob = RSA(our_public_key=keypair_bob[0], our_private_key=keypair_bob[1])

keypair_charlie = generate_rsa_keypair(2048)
rsa_charlie = RSA(our_public_key=keypair_charlie[0], our_private_key=keypair_charlie[1])

In [4]:
sender_shared_keypair = generate_rsa_keypair(2048)
# we now share the newly generated keypair with Alice, Bob, and Charlie. This is done by encrypting the private key of the sender with the public keys of Alice, Bob, and Charlie.
d_int = sender_shared_keypair[1].d
n_int = sender_shared_keypair[1].n
to_encrypt = d_int.to_bytes(256, byteorder='big') + n_int.to_bytes(256, byteorder='big')

encrypted_for_alice = rsa_sender.encrypt(to_encrypt, rsa_alice.our_public_key)
encrypted_for_bob = rsa_sender.encrypt(to_encrypt, rsa_bob.our_public_key)
encrypted_for_charlie = rsa_sender.encrypt(to_encrypt, rsa_charlie.our_public_key)

print("Encrypted for Alice:", encrypted_for_alice)
print("Encrypted for Bob:", encrypted_for_bob)
print("Encrypted for Charlie:", encrypted_for_charlie)

alice_decrypted = rsa_alice.decrypt(encrypted_for_alice)
bob_decrypted = rsa_bob.decrypt(encrypted_for_bob)
charlie_decrypted = rsa_charlie.decrypt(encrypted_for_charlie)

print("Alice decrypted:", alice_decrypted)
print("Bob decrypted:", bob_decrypted)
print("Charlie decrypted:", charlie_decrypted)

print("all match ?", alice_decrypted == bob_decrypted == charlie_decrypted == to_encrypt)

Encrypted for Alice: b'U\xa7\xcd\x99\x87M\xee\xbe`c/+%\x19\x1aE\xd0\xb6=\x8d&\xe5\xff\x8d\x98n\xc36-p\xbdU:\xf8"9\xb8\xa6eF\xbe\xf8(W\xfb\xb0\xf5c\xe9\xff\xd6F\xa3\xf8\xf7\xd2\x8d\xce\xbc\xfbhjn\xa3o\xed\xf5%\xb4|\x97\xef\xba\x07\x0b\x06\xeer\xc9\xdf\x9d\x9aw\xd0\x04\x9c\xe5\xe8\xc9\r\xff%1,\xbf\x84G\x95\xc8\x9bN>@\x18\xf7\x0b\xd2\xa5\xa2\xecdDYY\x10@\x9dLE\x88,\x7ff\x15\xf0\xee\x16\x80\r1\x89\xf4\xc8C\x08\x04Lb\x00\x8a\x07\xf3\xa9\x83\xd6\x9a\xea%q\xbe\xff\xd5\xe4\xdc\xf60\x00\x08\xbd\xae\xf8`\t\x833U\x8a\xa2p\xcd\xf4\x04\xc12\xcf\xee9A\xc8\xcf\xe5\x890F\xd9\xb3S\x95\xe9f\xee\x88\xeb1P(\xee\x95\xfb\xd3\xdcz\xc4\xaaW\xfaH\xe7=(\xe0\x1e\x0e\x9e\xdcK;\xd48\xda\x05\x10]\xf9\xca\x0f\xa6\xa5JYy\x9c\x0f1\xf3\x90\xe8(~\xfd\x8f\x9f\xca\xd7\xa3\t\xf1\xca\xc2\x14\xdb\x7f`X\xbb\xba\x91_K\x94\xd4C\x02\x95\xcd\xbe\xa4\xbb\xc6\xc4p\xdf\xd8&\xca\xdd&\xc7\xb0N\x87\xb3\t\x0c\x17\xd1\xbd\x8an\xb1\xd7\xe4\x04t\x1d\n\xbb,nyxzuBx\nz\xd2\xb0\x90\xc0\xfc\x9f\'\xdc\xbe\x02\xe8\x16\xd7\xee\xc7\x06\xa5\x04.\x17

In [5]:
# we now reconstruct the private key from the decrypted bytes
d_length = len(alice_decrypted) // 2
alice_receiver_private_key = RSAPrivateKey(n=int.from_bytes(alice_decrypted[d_length:], byteorder='big'), d=int.from_bytes(alice_decrypted[:d_length], byteorder='big'))
alice_receiver_public_key = RSAPublicKey(n=alice_receiver_private_key.n, e=keypair_alice[0].e)
alice_receiver_rsa = RSA(our_public_key=alice_receiver_public_key, our_private_key=alice_receiver_private_key)

# we could do the exact same thing for Bob and Charlie, but we will just use Alice's reconstructed keypair for the next steps.

msg = b"Lorem ipsum dolor sit amet, consectetur adipiscing elit. Sed do eiusmod tempor incididunt ut labore et dolore magna aliqua. Ut enim ad minim veniam, quis nostrud exercitation ullamco laboris nisi ut aliquip ex ea commodo consequat. Duis aute irure dolor in reprehenderit in voluptate velit esse cillum dolore eu fugiat nulla pariatur. Excepteur sint occaecat cupidatat non proident, sunt in culpa qui officia deserunt mollit anim id est laborum."
# we now encrypt the message with the sender's shared public key
encrypted_msg = rsa_sender.encrypt(msg, sender_shared_keypair[0])
print("Encrypted message:", encrypted_msg)
alice_decrypted_msg = alice_receiver_rsa.decrypt(encrypted_msg)
print("Alice decrypted message:", alice_decrypted_msg)
print("Message match ?", alice_decrypted_msg == msg)
print("Message length:", len(msg), "bytes")

Encrypted message: b'\xbb\x84\x85\xb1\xeeo7\xba\xe3&\xb2\x9e*J\x10\xbe\xf8k\x1dV\x13\xff.\xac\x17x\xc7!\xaf\x07\xc2\xdd\xbe\\\x96-k<\xbe\xc2\xc7\x945\'\x8e\xe7HP\xda\xb81\xf5\xbe\\\xae\x1d\x84\xc13\xecq:\xf4\xe8\xb1\x9f\x8a:\xf7Z\xff\xe8\xfbu\xa6\xa1s\x0c=4q\xffV/\xe2\x18\xbd\xbc\'\xa8\x1eU|s\xc5\xdc#\xccT \xf5\xb5!\xc9(\xe4\x81)\x91O\xaea\x042)\x85\xdcWM#\xac\\GGk\xa1w\x8e\x04*\xe3\xae\xd5^\t\xd7)<\x03\x1f,\xde\xc4\xc8\xd4\x0b\xb2=r\xc7\x8f\xfa+\xf3\xd4\x07[\xa7q\xd8\xdbw\x9fz\xd3\xd0o\xa3\xef#e\xc5\x98\r\xe7#\xcd\x15 y\xdd\x9b\x9e\xed\x89\x18\xef\x8a\xad\x91\x9a\x9f\xaa0\xa3\x1c\x84e\'\\\x80x\xa4\xa7\xe9\rYXd\xa8\x95\xe2dm\x96\x81\xde\xa2\xdc\xbbT\x17Dq\xad\x02J\x93\xce_}\xaa\xf7\xce\xd8pe*K\xbc\xa0\xdb5\xf3\x84%\xeauMDM\xb0\xf5\xd2\xdd/7\xcf\xad\xc4\xb1\x05\x10q;;\xabm#\xb1\x19\xc0\xcd\xe9\xb7\x99~\xc2\xc4"\x851&T\x12\x9d\xa8\xd7\xdbHXx\xbe\xa6?\xba\xf7<y\xb6\x15\xffT#\xb4\x82\xc3!\xcb\xcc\xa8\x95\xde\xc7\xb1J\xa4SP\x02\xe4\xa0\x02`\x8e\xbe\xfe#\x8f\x96\x99\x86>\xees\xcc\xb3\x04\xb7